In [1]:
import os
# Define your permanent server paths
DATA_DIR = "/data/jewettm/dynamic/datasets"
VIDEO_DIR = os.path.join(DATA_DIR, "Videos") # Or whatever the unzipped video folder name is
FILE_LIST_PATH = os.path.join(DATA_DIR, "FileList.csv")
VOLUME_TRACINGS_PATH = os.path.join(DATA_DIR, "VolumeTracings.csv")

print("Server paths initialized successfully!")
print(f"Checking FileList access: {os.path.exists(FILE_LIST_PATH)}")

Server paths initialized successfully!
Checking FileList access: True


# 2. Imports and Global Settings
Imports all necessary libraries (PyTorch, OpenCV, pandas, NumPy, etc.) and defines global constants: device (GPU if available), batch size, and image size.

Includes setup for mixed precision.

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import pandas as pd
import numpy as np
import cv2
import os
from scipy.spatial.distance import directed_hausdorff
from tqdm import tqdm
from torch.amp import autocast, GradScaler

# Global Settings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler('cuda')
BATCH_SIZE = 32
IMAGE_SIZE = 112

# 3. Pre-extract Video Frames (Cache)
The extraction will take 30–60 minutes for ~10,000 videos (each about 30–100 frames). The progress bar will show per‑video progress.

After extraction, training per epoch will drop from ~20 minutes to 2–4 minutes.

Defines a function that extracts every frame from all .mp4 videos, resizes to 112×112, converts RGB, and saves as NumPy .npy files. Runs extraction once unless cache already exists.

In [4]:
def extract_all_frames(video_dir, output_dir, target_size=(112,112)):
    if os.path.exists(output_dir) and any(f.endswith('.npy') for f in os.listdir(output_dir)):
        print(f"Cache already exists at {output_dir}. Skipping extraction.")
        return
    """
    Extract all frames from all videos and save as .npy files.
    Args:
        video_dir: folder containing video files (.mp4 or .avi)
        output_dir: folder where .npy files will be saved (one per frame)
        target_size: (H, W) to resize frames
    """
    os.makedirs(output_dir, exist_ok=True)
    # Check for both standard EchoNet formats (.avi and .mp4)
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4') or f.endswith('.avi')]
    print(f"Found {len(video_files)} videos. Extracting frames...")

    for video_file in tqdm(video_files, desc="Videos"):
        video_name = os.path.splitext(video_file)[0]  # Removes extension cleanly
        video_path = os.path.join(video_dir, video_file)
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Warning: Cannot open {video_path}")
            continue
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, target_size)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            np.save(os.path.join(output_dir, f"{video_name}_{frame_idx:04d}.npy"), frame)
            frame_idx += 1
        cap.release()
        
    print(f"Extraction complete. Files saved in {output_dir}")

# ---- Run extraction ----
# Direct, permanent paths on the GPU server (No Google Drive mimicking)
extract_all_frames(
    video_dir='/data/jewettm/dynamic/datasets/Videos', 
    output_dir='/data/jewettm/dynamic/datasets/video_frames_cache'
)

Found 10030 videos. Extracting frames...


Videos: 100%|██████████| 10030/10030 [06:49<00:00, 24.49it/s]

Extraction complete. Files saved in /data/jewettm/dynamic/datasets/video_frames_cache


# 4. Custom Dataset via Segmentation Parser
Defines EchoSegmentationDataset that:
* Reads the CSV tracings.
* For a given frame, reconstructs the left ventricle mask from all (X1,Y1,X2,Y2) point pairs.
* Loads frames either from the fast .npy cache or directly from videos (with fallback).
* Applies transforms and returns (image, mask) tensors.





In [5]:
class EchoSegmentationDataset(Dataset):
    def __init__(self, root_dir, file_list_csv, tracings_csv, transform=None, cache_dir=None):
        self.root_dir = root_dir
        self.transform = transform
        self.cache_dir = cache_dir
        self.img_size = IMAGE_SIZE

        print("Loading CSV files...")
        self.file_list = pd.read_csv(file_list_csv)
        self.tracings = pd.read_csv(tracings_csv)

        # Pre-group coordinates into a dictionary to eliminate the Pandas training loop bottleneck
        print("Optimizing tracing coordinates map for speed...")
        self.points_dict = {}
        for (video_name, frame_idx), group in self.tracings.groupby(['FileName', 'Frame']):
            pts = group[['X1', 'Y1', 'X2', 'Y2']].values.reshape(-1, 2).astype(np.int32)
            self.points_dict[(video_name, frame_idx)] = pts

        self.frame_list = list(self.points_dict.keys())

        # If not using cache, filter to frames whose video file exists
        if self.cache_dir is None:
            original_len = len(self.frame_list)
            filtered_list = []
            for video_name, frame_idx in self.frame_list:
                base_name = os.path.splitext(video_name)[0]
                video_path = os.path.join(self.root_dir, f"{base_name}.mp4")
                if os.path.exists(video_path):
                    filtered_list.append((video_name, frame_idx))
            self.frame_list = filtered_list
            print(f"Filtered dataset: {len(self.frame_list)} frames with existing videos (was {original_len})")

    def __len__(self):
        return len(self.frame_list)

    def __getitem__(self, idx):
        video_name, frame_idx = self.frame_list[idx]
        base_name = os.path.splitext(video_name)[0]

        # ----- Load frame -----
        frame = None
        if self.cache_dir is not None:
            frame_path = os.path.join(self.cache_dir, f"{base_name}_{frame_idx:04d}.npy")
            try:
                frame = np.load(frame_path)
            except FileNotFoundError:
                pass

        # Fallback if cache is missing or not used
        if frame is None:
            video_path = os.path.join(self.root_dir, f"{base_name}.mp4")
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                frame = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
            else:
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                ret, frame = cap.read()
                if not ret:
                    cap.release()
                    cap = cv2.VideoCapture(video_path)
                    for _ in range(frame_idx):
                        ret = cap.grab()
                        if not ret:
                            break
                    ret, frame = cap.read()
                cap.release()
                
                if not ret:
                    frame = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
                else:
                    frame = cv2.resize(frame, (self.img_size, self.img_size))
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # ----- Reconstruct mask -----
        pts = self.points_dict.get((video_name, frame_idx), np.array([], dtype=np.int32))
        mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
        if len(pts) >= 3:
            cv2.fillPoly(mask, [pts], 1)

        # ----- Apply transforms -----
        if self.transform:
            image = self.transform(frame)
            mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
            return image, mask
        else:
            frame_tensor = torch.tensor(frame, dtype=torch.float32).permute(2, 0, 1) / 255.0
            mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
            return frame_tensor, mask_tensor

# 5. Model Definition
Creates a DeepLabV3+ segmentation model with a ResNet50 backbone, pretrained on COCO. Adjusts the final classifier to output a single channel (binary LV mask). Added Dropout2d(0.2) in the classifier head. Updated to consider dropout for regularisation.

In [6]:
def get_deeplabv3p_model(num_classes=1):
    # Using ResNet50 as the backbone for a balance of speed and accuracy
    model = models.segmentation.deeplabv3_resnet50(weights='DEFAULT')
    model.classifier[-1] = nn.Sequential(nn.Dropout2d(0.2),   # 20% dropout
        nn.Conv2d(256, num_classes, kernel_size=1)
    )
    return model.to(device)

# 6. Training and Eval Functions
Contains:
* train_epoch – single training epoch with progress bar.
* evaluate_model – computes Dice, IoU, pixel accuracy, and Hausdorff distance on a given dataloader.
* compute_hausdorff – helper for the Hausdorff metric.

Added DiceBCELoss, WeightedDiceBCELoss (for future testing), morphological closing in evaluate_model, and postprocess_mask

In [7]:
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        bce = self.bce(pred, target)
        pred_sigmoid = torch.sigmoid(pred)
        intersection = (pred_sigmoid * target).sum()
        dice = 1 - (2 * intersection + self.smooth) / (pred_sigmoid.sum() + target.sum() + self.smooth)
        return bce + dice

# Weighted Dice for later experimenting with eval improvements
class WeightedDiceBCELoss(nn.Module):
    def __init__(self, pos_weight=1.0, smooth=1e-6):
        super().__init__()
        self.pos_weight = pos_weight
        self.smooth = smooth

    def forward(self, pred, target):
        pos_weight_tensor = torch.tensor([self.pos_weight]).to(pred.device)
        bce = nn.functional.binary_cross_entropy_with_logits(pred, target, pos_weight=pos_weight_tensor)
        pred_sigmoid = torch.sigmoid(pred)
        intersection = (pred_sigmoid * target).sum()
        dice = 1 - (2 * intersection + self.smooth) / (pred_sigmoid.sum() + target.sum() + self.smooth)
        return bce + dice
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss=0.0
    pbar = tqdm(loader, desc="Training")
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        # Enable automatic mixed precision
        with autocast(device_type='cuda'):
            output = model(images)['out']
            loss = criterion(output, masks)

        # Scale loss, backpropagate, and step
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)   # sum batch loss
        pbar.set_postfix({"loss": loss.item()})
    return total_loss / len(loader.dataset)

def evaluate_model(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    dices, ious, accuracies = [], [], []
    with torch.no_grad():
        pbar = tqdm(loader, desc="Evaluating")
        for images, masks in pbar:
            images, masks = images.to(device), masks.to(device)
            output = model(images)['out']
            pred_bin = (torch.sigmoid(output) > 0.5).float()

            # --- Apply morphological closing to predictions ---
            pred_np = pred_bin.cpu().numpy()          # shape (B,1,H,W)
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
            for i in range(pred_np.shape[0]):
                pred_np[i, 0] = cv2.morphologyEx(pred_np[i, 0], cv2.MORPH_CLOSE, kernel)
            pred_bin = torch.from_numpy(pred_np).to(device)

            # Pixel accuracy
            acc = (pred_bin == masks).float().mean().item()
            accuracies.append(acc)

            # Dice & IoU
            inter = (pred_bin * masks).sum()
            dice = (2 * inter) / (pred_bin.sum() + masks.sum() + 1e-6)
            union = pred_bin.sum() + masks.sum() - inter
            iou = inter / (union + 1e-6)
            dices.append(dice.item())
            ious.append(iou.item())

            # Collect for Hausdorff
            all_preds.append(pred_bin.cpu())
            all_targets.append(masks.cpu())

            pbar.set_postfix({"dice": dice.item()})

    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    hd = compute_hausdorff(all_preds, all_targets)
    return np.mean(dices), np.mean(ious), hd, np.mean(accuracies)

def compute_hausdorff(preds, targets):
    # preds, targets shape: (N, 1, H, W)
    hd_list = []
    for i in range(preds.shape[0]):
        p = np.column_stack(np.where(preds[i,0] > 0))
        t = np.column_stack(np.where(targets[i,0] > 0))
        if len(p) == 0 or len(t) == 0:
            hd_list.append(0.0)
        else:
            hd = max(directed_hausdorff(p, t)[0], directed_hausdorff(t, p)[0])
            hd_list.append(hd)
    return np.mean(hd_list)   # mean HD over all frames

def postprocess_mask(pred_bin, min_area=100):
    pred_np = pred_bin.cpu().numpy()[0,0].astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    closed = cv2.morphologyEx(pred_np, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            cv2.drawContours(closed, [cnt], -1, 0, -1)
    return torch.from_numpy(closed).unsqueeze(0).unsqueeze(0).float()

# 7. Compute Dataset Statistics & Build Full Dataset
Creates a temporary dataset without normalization, computes mean and standard deviation from 1000 random frames, then builds the final dataset with custom normalization. Creates separate train_transforms (with augmentation) and val_transforms. The temporary dataset and mean/std calculation remain, but the final datasets use different transforms

In [8]:
# ---- Step 1: Create a temporary dataset without normalization ----
temp_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

# SERVER PATHS: Cleaned and targeted to permanent directory structures
SERVER_ROOT = '/data/jewettm/dynamic/datasets'

temp_dataset = EchoSegmentationDataset(
    root_dir=f'{SERVER_ROOT}/Videos',
    file_list_csv=f'{SERVER_ROOT}/FileList.csv',
    tracings_csv=f'{SERVER_ROOT}/VolumeTracings.csv',
    transform=temp_transform,
    cache_dir=f'{SERVER_ROOT}/video_frames_cache'
)

# Compute mean and std (unchanged)
def compute_dataset_stats(dataset, num_samples=1000):
    loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0)
    mean = 0.0
    std = 0.0
    for i, (img, _) in enumerate(loader):
        if i >= num_samples:
            break
        mean += img.mean(dim=[0,2,3]).squeeze()
        std += img.std(dim=[0,2,3]).squeeze()
    mean /= (i+1)
    std /= (i+1)
    return mean.tolist(), std.tolist()

data_mean, data_std = compute_dataset_stats(temp_dataset)
print(f"Custom mean: {data_mean}, std: {data_std}")

# ---- Step 3: Create separate transforms for training and validation ----
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=data_mean, std=data_std)
])

val_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=data_mean, std=data_std)
])

# ---- Step 4: Create datasets with appropriate transforms ----
train_full_dataset = EchoSegmentationDataset(
    root_dir=f'{SERVER_ROOT}/Videos',
    file_list_csv=f'{SERVER_ROOT}/FileList.csv',
    tracings_csv=f'{SERVER_ROOT}/VolumeTracings.csv',
    transform=train_transforms,
    cache_dir=f'{SERVER_ROOT}/video_frames_cache'
)

val_full_dataset = EchoSegmentationDataset(
    root_dir=f'{SERVER_ROOT}/Videos',
    file_list_csv=f'{SERVER_ROOT}/FileList.csv',
    tracings_csv=f'{SERVER_ROOT}/VolumeTracings.csv',
    transform=val_transforms,
    cache_dir=f'{SERVER_ROOT}/video_frames_cache'
)

Loading CSV files...
Optimizing tracing coordinates map for speed...
Custom mean: [0.1278531402349472, 0.12870950996875763, 0.13031411170959473], std: [0.1868864744901657, 0.18816988170146942, 0.18957778811454773]
Loading CSV files...
Optimizing tracing coordinates map for speed...
Loading CSV files...
Optimizing tracing coordinates map for speed...


# 8. Train/Validation Split
Splits the dataset by unique video names (80% train / 20% validation) to prevent data leakage. Creates PyTorch Subset objects.

In [9]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# Get unique video names from the training dataset (same as before)
unique_videos = list(set([name for name, _ in train_full_dataset.frame_list]))
train_videos, val_videos = train_test_split(unique_videos, test_size=0.2, random_state=42)

train_indices = [i for i, (name, _) in enumerate(train_full_dataset.frame_list) if name in train_videos]
val_indices   = [i for i, (name, _) in enumerate(val_full_dataset.frame_list) if name in val_videos]

train_dataset = Subset(train_full_dataset, train_indices)
val_dataset   = Subset(val_full_dataset, val_indices)

print(f"Train frames: {len(train_dataset)}, Val frames: {len(val_dataset)}")

Train frames: 16040, Val frames: 4010


# 9. DataLoaders, Training Loop, & Model Saving
Instantiates train and validation loaders, initialises the DeepLabV3+ model, Adam optimizer, and binary cross‑entropy loss. Uses ReduceLROnPlateau scheduler, early stopping with patience, class‑imbalance computation (though unused), and final evaluation on full validation set. Epochs train on training set, evaluate on validation set, prints metrics each epoch. Saves the final model weights to Google Drive.

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import random
from torch.utils.data import Subset

# Setup a clean local tracking folder
OUTPUT_DIR = "/data/jewettm/dynamic/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Compute class imbalance
print("Computing class imbalance...")
pos = 0
neg = 0
for i, (_, mask) in enumerate(train_loader):
    pos += (mask == 1).sum().item()
    neg += (mask == 0).sum().item()
    if i > 50:
        break
if pos > 0:
    pos_weight_val = neg / pos
    print(f"Positive ratio: {pos/(pos+neg):.4f}")
    print(f"Suggested pos_weight: {pos_weight_val:.2f}")
else:
    pos_weight_val = 1.0

model = get_deeplabv3p_model(num_classes=1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = WeightedDiceBCELoss(pos_weight=15.01)

num_val_samples = 500
indices = random.sample(range(len(val_dataset)), min(num_val_samples, len(val_dataset)))

scaler = GradScaler('cuda')
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

epochs = 30
best_dice = 0.0
patience = 7
epochs_no_improve = 0
epoch_losses = []
epoch_dices = []
epoch_ious = []
epoch_hds = []
epoch_accs = []

for epoch in range(1, epochs + 1):
    avg_loss = train_epoch(model, train_loader, optimizer, criterion, scaler)
    dice, iou, hd, acc = evaluate_model(model, val_loader)
    epoch_losses.append(avg_loss)
    epoch_dices.append(dice)
    epoch_ious.append(iou)
    epoch_hds.append(hd)
    epoch_accs.append(acc)
    print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | Val Dice: {dice:.4f} | IoU: {iou:.4f} | HD: {hd:.2f} | Acc: {acc:.4f}")

    scheduler.step(dice)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"  Learning rate: {current_lr:.6f}")

    # Save best model to our server output folder
    if dice > best_dice:
        best_dice = dice
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_lv_seg_model.pth')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping after {epoch} epochs (best Dice: {best_dice:.4f})")
            break

# Print a summary table of all epochs
print("\n" + "="*80)
print("EPOCH SUMMARY TABLE")
print("="*80)
print(f"{'Epoch':<6} {'Loss':<8} {'Val Dice':<10} {'IoU':<8} {'HD':<8} {'Acc':<8}")
print("-"*80)
for i in range(len(epoch_losses)):
    print(f"{i+1:<6} {epoch_losses[i]:<8.4f} {epoch_dices[i]:<10.4f} {epoch_ious[i]:<8.4f} {epoch_hds[i]:<8.2f} {epoch_accs[i]:<8.4f}")
print("="*80)

# Load best model for final use
model.load_state_dict(torch.load(f'{OUTPUT_DIR}/best_lv_seg_model.pth'))

print("\n=== Final evaluation on full validation set ===")
model.eval()

dice_full, iou_full, hd_full, acc_full = evaluate_model(model, val_loader)
print(f"Full Val Dice: {dice_full:.4f} | IoU: {iou_full:.4f} | Hausdorff: {hd_full:.2f} | Acc: {acc_full:.4f}")

with open(f'{OUTPUT_DIR}/final_metrics.txt', 'w') as f:
    f.write(f"Dice: {dice_full:.4f}\nIoU: {iou_full:.4f}\nHausdorff: {hd_full:.2f}\nAccuracy: {acc_full:.4f}\n")

Computing class imbalance...
Positive ratio: 0.0625
Suggested pos_weight: 15.01
Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /home/jewettm/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth


100%|██████████| 161M/161M [00:03<00:00, 53.9MB/s] 
Evaluating: 100%|██████████| 126/126 [00:04<00:00, 29.83it/s, dice=0.747]


Epoch 1 | Loss: 0.4782 | Val Dice: 0.6988 | IoU: 0.5373 | HD: 8.36 | Acc: 0.9606
  Learning rate: 0.001000


Evaluating: 100%|██████████| 126/126 [00:04<00:00, 29.58it/s, dice=0.783]


Epoch 2 | Loss: 0.4008 | Val Dice: 0.7410 | IoU: 0.5888 | HD: 6.80 | Acc: 0.9646
  Learning rate: 0.001000


...

Epoch 14 | Loss: 0.3587 | Val Dice: 0.7512 | IoU: 0.6017 | HD: 5.75 | Acc: 0.9661
  Learning rate: 0.000250


Evaluating: 100%|██████████| 126/126 [00:04<00:00, 28.96it/s, dice=0.793]


Epoch 15 | Loss: 0.3546 | Val Dice: 0.7554 | IoU: 0.6072 | HD: 5.47 | Acc: 0.9664
  Learning rate: 0.000250

EPOCH SUMMARY TABLE
Epoch  Loss     Val Dice   IoU      HD       Acc     
--------------------------------------------------------------------------------
1      0.4782   0.6988     0.5373   8.36     0.9606  
2      0.4008   0.7410     0.5888   6.80     0.9646  
3      0.3881   0.7365     0.5831   6.40     0.9633  
4      0.3808   0.7520     0.6028   5.31     0.9646  
5      0.3782   0.7503     0.6006   5.80     0.9661  
6      0.3739   0.7456     0.5945   6.13     0.9651  
7      0.3725   0.7478     0.5974   6.02     0.9653  
8      0.3644   0.7548     0.6064   5.46     0.9654  
9      0.3624   0.7512     0.6017   5.72     0.9661  
10     0.3619   0.7545     0.6060   5.61     0.9660  
11     0.3609   0.7558     0.6077   5.47     0.9663  
12     0.3604   0.7514     0.6019   5.90     0.9657  
13     0.3595   0.7509     0.6014   6.01     0.9656  
14     0.3587   0.7512     0.6017 

Evaluating: 100%|██████████| 126/126 [00:04<00:00, 29.25it/s, dice=0.791]


Full Val Dice: 0.7558 | IoU: 0.6077 | Hausdorff: 5.47 | Acc: 0.9663
